# Sample run for Fisher Market

In [1]:
using Pkg
Pkg.activate("../")

  Activating project at `~/workspace/ExchangeMarket.jl/scripts`


In [2]:
using Revise
using Random, SparseArrays, LinearAlgebra
using JuMP, MosekTools
using Plots, LaTeXStrings, Printf
import MathOptInterface as MOI

using ExchangeMarket

include("../tools.jl")
include("../plots.jl")
include("./setup.jl")
switch_to_pdf(; bool_use_html=false)

:pdf

In [3]:
Random.seed!(1)
n = 20
m = 100
ρ = 1.0
# method_filter(name) = name ∈ [:LogBar]
method_filter(name) = name ∈ [:Tât, :PropRes]
method_filter(name) = name ∈ [:PropRes]

f0 = FisherMarket(m, n; ρ=1.0, bool_unit=true, scale=30.0, sparsity=0.9, ε_br_play=1e-1)
linconstr = LinearConstr(1, n, ones(1, n), [sum(f0.w)])
ρfmt = @sprintf("%+.2f", ρ)
σfmt = @sprintf("%+.2f", f0.σ[1])


FisherMarket initialization started...
FisherMarket cost matrix initialized in 0.0523 seconds
FisherMarket initialized in 0.0814 seconds


"+Inf"

In [4]:
alg = Conic(n, m)
ff = copy(f0)
ExchangeMarket.__create_dual(alg, ff)

FisherMarket initialization started...
FisherMarket cost matrix initialized in 0.0002 seconds
FisherMarket initialized in 0.0002 seconds
Problem
  Name                   :                 
  Objective sense        : minimize        
  Type                   : CONIC (conic optimization problem)
  Constraints            : 2000            
  Affine conic cons.     : 100 (300 rows)
  Disjunctive cons.      : 0               
  Cones                  : 0               
  Scalar variables       : 220             
  Matrix variables       : 0               
  Integer variables      : 0               

Optimizer started.
Presolve started.
Linear dependency checker started.
Linear dependency checker terminated.
Eliminator started.
Freed constraints in eliminator : 0
Eliminator terminated.
Eliminator - tries                  : 1                 time                   : 0.00            
Lin. dep.  - tries                  : 1                 time                   : 0.00            
Lin. dep.  - 

In [5]:
table_time = []
results = []
results_phi = Dict()
results_ground = Dict()

# -----------------------------------------------------------------------
# compute ground truth
# -----------------------------------------------------------------------
# μ₀ = 1e1
f1 = copy(f0)
p₀ = ones(n) * sum(f1.w) ./ (n)
# p₀ = ones(n) .* μ₀
x₀ = ones(n, m) ./ m
f1.x .= x₀
f1.p .= p₀;
# use proportional response method to compute ground truth
(name, method, kwargs) = method_kwargs[end]
alg = method(
    n, m, copy(p₀);
    linconstr=linconstr,
    # μ=μ₀,
    kwargs...
)
traj = opt!(
    alg, f1;
    # loginterval=1,
    keep_traj=true,
    bool_init_phase=true,
    maxiter=2000
)
results_ground[ρ] = (alg, traj, f1)
results_phi[ρ] = copy(alg.p)

FisherMarket initialization started...
FisherMarket cost matrix initialized in 0.0001 seconds
FisherMarket initialized in 0.0002 seconds
-----------------------------------------------------------------------------------
              ExchangeMarket.jl: A Julia Package for Exchange Market              
                               © Chuwen Zhang (2024)                               
-----------------------------------------------------------------------------------
 subproblem solver alias       := Bids
 subproblem solver style       := bids
 option for step               := shmyrev
 option for step size          := cc13
--------------------------------------------------------------------------------------------
      k |             φ |    |∇φ| |    |Δp| |      Δu |       t |      tₗ |      α 
      0 | +5.354240e-01 | 0.0e+00 | 1.5e-02 | 1.1e+01 | 5.4e-01 | 5.4e-01 | 5.0e+00 
     20 | +2.885197e+00 | 0.0e+00 | 1.8e-04 | 3.6e-01 | 5.9e-01 | 5.9e-01 | 5.0e+00 
     40 | +2.897528e+0

20-element Vector{Float64}:
 0.049484614757634146
 0.048060292612044954
 0.052600545603158
 0.051510137857614366
 0.05120199848373927
 0.05001877661202772
 0.04999491157388442
 0.051354385439391095
 0.04806741530183804
 0.050357461763826404
 0.05102673236494335
 0.05166722759070208
 0.051110176894986564
 0.048837359959969066
 0.05087638396915393
 0.049759060656819856
 0.047953719879894885
 0.05217083226611466
 0.04821652781166095
 0.04573143860059636

In [6]:
[ff.val_u f1.val_u]

100×2 Matrix{Float64}:
  3.96106    3.96191
  1.01575    1.01594
  9.0917     9.09386
  5.4763     5.4769
 10.4451    10.4483
  6.40678    6.40727
  6.18155    6.18271
  0.506928   0.506975
  1.17461    1.17524
  9.88266    9.88465
  ⋮         
  8.59951    8.60016
  0.879743   0.879608
  2.07653    2.07764
  4.2617     4.261
  1.04009    1.04034
  3.31428    3.31453
  1.90117    1.90158
  5.80226    5.80321
  3.24676    3.247

# Validate LSE

- use ForwardDiff to test grad and Hessian
- create a copy f1 and run Optim using Interior-Point (need to p ≥ 0 )

## Validate gradient and Hessian with ForwardDiff

In [ ]:
include("./linear.jl")

In [ ]:
f1 = copy(f0)
p_test = ones(n) * sum(f1.w) / n
t_val = 1e-3

if n < 100
    # analytic gradient
    g_analytic = zeros(n)
    lse_grad!(g_analytic, p_test, f1; t=t_val)

    # ForwardDiff gradient
    g_fd = ForwardDiff.gradient(p -> lse_dual(p, f1; t=t_val), p_test)

    @show norm(g_analytic - g_fd)
    @assert norm(g_analytic - g_fd) < 1e-8 "Gradient mismatch!"

    # analytic Hessian (affine-scaled: P ∇²φ P)
    H_analytic = zeros(n, n)
    lse_hess!(H_analytic, p_test, f1; t=t_val)

    # ForwardDiff Hessian (plain ∇²φ), then apply affine scaling
    H_fd = ForwardDiff.hessian(p -> lse_dual(p, f1; t=t_val), p_test)
    for l in 1:n, k in 1:n
        H_fd[k, l] *= p_test[k] * p_test[l]
    end

    @show norm(H_analytic - H_fd)
    @assert norm(H_analytic - H_fd) < 1e-6 "Hessian mismatch!"

    # verify matrix-free matvec against dense H
    v_test = randn(n)
    Hv_dense = H_analytic * v_test
    Hv_matvec = zeros(n)
    lse_hess_matvec!(Hv_matvec, v_test, p_test, f1; t=t_val)

    @show norm(Hv_dense - Hv_matvec)
    @assert norm(Hv_dense - Hv_matvec) < 1e-8 "Matvec mismatch!"

    println("✓ Gradient, Hessian, and matvec match.")
end

In [ ]:
sparse(H_analytic)

## Check Hessian Approx.

In [ ]:
# exact affine-scaled Hessian: H = P ∇²φ P
f1 = copy(f0)
p_test = ones(n) * sum(f1.w) / n
t_val = 1e-2

# --- diagonal-only check ---
H_diag = zeros(n, n)
lse_hess!(H_diag, p_test, f1; t=t_val, bool_diag_only=true)
Λ_exact = diag(H_diag)

res_diag = lse_hess_dr1(p_test, f1; t=t_val, bool_diag_only=true)
@show norm(Λ_exact - res_diag.Λ)
@assert norm(Λ_exact - res_diag.Λ) < 1e-10 "Diagonal mismatch!"
println("✓ Diagonal matches exactly.")

# --- full check ---
H = zeros(n, n)
lse_hess!(H, p_test, f1; t=t_val)
(; Λ, Σ, ξ, Ω, matG, matD, matE, vecL) = lse_hess_dr1(p_test, f1; t=t_val, bool_full=true)

# verify Σ = (matE .^ 2) * β
β = [f1.w[i] * (1.0 + vecL[i] / t_val) for i in 1:f1.m]
@show norm(Σ - (matE .^ 2) * β)

# exact rank-one sum: V = matE * diag(β) * matEᵀ
# verify: Λ - V = H
Vv = matE * diagm(β) * matE'
@show norm(diagm(Λ) - Vv - H) / norm(H)

# DR1 approximation: H̃ = diag(Λ) - Ω ξξᵀ
H_dr1 = diagm(Λ) - Ω .* (ξ * ξ')
@show norm(H - H_dr1) / norm(H)

# Diagonal approximation: Ĥ = diag(Λ - Σ)
H_diag_approx = diagm(Λ .- Σ)
@show norm(H - H_diag_approx) / norm(H)

# sparsified V
Vs = droptol!(sparse(Vv), 1e-10)
@show nnz(Vs) / length(Vv)

# eigenvalue comparison
@show eigvals(H) |> extrema
@show eigvals(H_dr1) |> extrema
@show eigvals(H_diag_approx) |> extrema

### Von Neumann Series

In [ ]:
# check {eq.lse.neumann.precond}
# H = Ĥ - R_off, where Ĥ = diag(Λ - Σ), R_off = Ĥ - H (off-diagonal part)
# H⁻¹ = Ĥ⁻¹ Σ_k (R_off Ĥ⁻¹)^k
# M_p⁻¹ = Ĥ⁻¹ Σ_{k=0}^p (R_off Ĥ⁻¹)^k
# Validate: ‖M_p⁻¹ - H⁻¹‖_F / ‖H⁻¹‖_F

# reuse from previous cells: H, Λ, Σ

# Ĥ = diag(Λ - Σ) and R_off = Ĥ - H
Hhat_diag = Λ .- Σ
Hhat_inv_diag = 1 ./ Hhat_diag
R_off = diagm(Hhat_diag) - H   # off-diagonal residual (note: H = Ĥ - R_off)

@show norm(R_off) / norm(H)

# reference inverse (pseudoinverse since H is rank-deficient)
Hinv = pinv(H)
normHinv = norm(Hinv)

# δ_off = ‖Ĥ⁻¹ R_off‖_F
Hhat_inv_R = diagm(Hhat_inv_diag) * R_off
δ_off = norm(Hhat_inv_R)
@printf("δ_off = ‖Ĥ⁻¹ R_off‖_F = %.6e\n", δ_off)

# Neumann preconditioners M_p⁻¹
R_Hinv = R_off * diagm(Hhat_inv_diag)  # R_off Ĥ⁻¹

println("─"^75)
@printf("  p │  ‖M_p⁻¹ - H⁺‖_F/‖H⁺‖_F │  ‖M_p⁻¹ - H⁺‖_F         │  ok?\n")
println("─"^75)
for p in 0:20
    # Σ_{k=0}^p (R_off Ĥ⁻¹)^k
    Neumann_sum = Matrix(1.0I, n, n)
    RHk = Matrix(1.0I, n, n)
    for k in 1:p
        RHk = RHk * R_Hinv
        Neumann_sum .+= RHk
    end
    M_p_inv = diagm(Hhat_inv_diag) * Neumann_sum
    residual = norm(M_p_inv - Hinv)
    rel_res = residual / normHinv
    ok = rel_res < 1.0
    @printf("  %d │  %.6e             │  %.6e             │  %s\n", p, rel_res, residual, ok ? "✓" : "✗")
end
println("─"^75)

In [ ]:
Rh = sparse(round.(R_Hinv; digits=1))
nnz(Rh)

# Solve LSE dual with IPNewton (p ≥ 0), pure Newton

In [ ]:
include("./linear.jl")
f2 = copy(f0)
p₀ = ones(f0.n) * sum(f2.w) / f0.n
t₀ = 5e0

res = lse_dual_newton(f2; t=t₀, p₀=p₀, hessian=:direct, maxiter=1200, gtol=1e-10)

@show norm(res.p - results_phi[ρ])

lse_allocation!(f2, res.p; t=t₀)
lse_alg = OptimAlg(res.p)
validate(f2, lse_alg)

In [ ]:
validate(f1, alg)
validate(f2, lse_alg)

In [ ]:
(alg.p - lse_alg.p) .|> abs |> maximum

## Approx. Newton iteration with backtracking

In [ ]:
include("./linear.jl")
f3 = copy(f0)
p₀ = ones(n) * sum(f3.w) / n

res_dr1 = lse_dual_newton(f3; t=1e-3, p₀=p₀, hessian=:vonneumann, maxiter=2000, gtol=1e-10, p_order=5)

@show norm(res_dr1.p - results_phi[ρ])

lse_allocation!(f3, res_dr1.p; t=1e-2)
lse_alg_dr1 = OptimAlg(copy(res_dr1.p))
validate(f3, lse_alg_dr1)

# Use regularized Newton (Mishchenko)

In [ ]:
include("./linear.jl")
f4 = copy(f0)
p₀ = ones(n) * sum(f4.w) / n
t₀ = 5e-1

res_rn = lse_dual_regnewton(f4; t=t₀, p₀=p₀, maxiter=1000, gtol=1e-10)

@show res_rn.σ
@show norm(res_rn.p - results_phi[ρ])

lse_allocation!(f4, res_rn.p; t=t₀)
lse_alg_rn = OptimAlg(res_rn.p)
validate(f4, lse_alg_rn)

# Path-Following Homotopy Newton

In [ ]:
include("./linear.jl")
f5 = copy(f0)
p₀ = ones(n) * sum(f5.w) / n
t₀ = 5e-1

res_pfh = lse_dual_pfh(
    f5; t=t₀, p₀=p₀, μ₀=1e-3, μ_factor=1e-1,
    hessian=:cg, p_order=6, maxiter=500, gtol=1e-7
)

@show res_pfh.μ
@show norm(res_pfh.p - results_phi[ρ])

lse_allocation!(f5, res_pfh.p; t=t₀)
lse_alg_pfh = OptimAlg(res_pfh.p)
validate(f5, lse_alg_pfh)

# Tranforming to NLP